# NB19 — LEARNING CURVE: DOES THE SMALL-SAMPLE (N = 30 EGGS) EXPLANATION HOLD? | NIR-HUEVOS 2026

**Post hoc, reviewer-driven. Uses the frozen splits and frozen per-fold configurations; no retuning, no new model selection.**

## Question
Reviewer 1 argues that the poor recurrent/convolutional results are "overwhelmingly likely" caused by the small number of independent biological units (N = 30),
not by the architecture. The manuscript can only discuss this verbally. NB19 gives an empirical handle: for every outer fold, each model is refitted on **random subsets
of the 24 outer-training eggs** (6, 12, 18 and all 24 eggs) and always evaluated on the same 6 unseen outer-test eggs.

* If the deep models improve steeply with more eggs while PLSR/SVR are already flat, the "N-limited" explanation gains support (and the wording about architecture must be softened).
* If the deep models stay far above PLSR/SVR across the whole 6→24 range, the sample-size explanation is not supported *within the range that can be examined*.
  (Extrapolation beyond 24 training eggs is **not** claimed either way.)

## Protocol
* Configurations are inherited per outer fold: PLSR/SVR from NB03, ANN/LSTM from NB04, `CNN1D` from NB11, `CNN1D_FLAT` from NB16. Integrity asserts confirm that the *final* NB03/NB04 files are used.
* Deep models: `epochs_used = round(frozen_epochs × 24 / n_eggs)` when `SCALE_EPOCHS=True`, which keeps the number of gradient updates roughly constant as the training set shrinks. A single seed (2026) is used per fit.
* Subsets are drawn without replacement with a deterministic seed per (fold, size, repetition); size 24 uses all eggs and is a single fit.
* Inference is at the egg level; whole-egg bootstrap (10,000 resamples, seed 20260915).

## Runtime (Colab GPU)
≈ 30–60 min with LSTM (`INCLUDE_LSTM = True`), ≈ 10–15 min without. Resumable: each fit is checkpointed.

In [1]:
import os, sys, json, gc, time, random, hashlib, platform, subprocess, shutil, warnings, re
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ------------------------------- switches -------------------------------
QUICK_TEST       = os.environ.get('NIR_QUICK_TEST', '0') == '1'   # smoke test only; NEVER use for reported results
STRICT_INTEGRITY = os.environ.get('NIR_STRICT', '1') == '1'       # verify SHA-256 of dataset and frozen split manifest
RUN_PART_A = False
RUN_PART_B = False
RUN_PART_C = False

# ------------------------------- paths ---------------------------------
PROJECT_ROOT = Path(os.environ.get('NIR_PROJECT_ROOT', '/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026'))
RAW_DIR   = PROJECT_ROOT / '01_DATA_RAW'
SPLIT_DIR = PROJECT_ROOT / '03_SPLITS_FROZEN'
RES_ROOT  = PROJECT_ROOT / '05_RESULTS'
RESULT_DIR = RES_ROOT / 'REVISION_REVIEWERS_2026_09' / 'NB19_LEARNING_CURVE'
CKPT_DIR   = RESULT_DIR / '_CHECKPOINT'
ZIP_DIR    = RES_ROOT / 'ZIP_PACKAGES'
for p in [RESULT_DIR, CKPT_DIR, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_FILE  = RAW_DIR / 'dataset_egg_storage_RAW.csv'
OUTER_FILE = SPLIT_DIR / 'outer_group_assignment_seed2026.csv'
SPLIT_MANIFEST_FILE = SPLIT_DIR / 'split_manifest.json'
EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'
EXPECTED_SPLIT_MANIFEST_SHA256 = 'fbeb8fa19d522cd91bee875bf5731cda264475da27bc7e93c25ca0d6f0f33717'

RUN_REVISION = 'NB19_v1_learning_curve_frozen_configurations'

# ------------------------- frozen protocol (= NB11) --------------------
N_OUTER, N_INNER = 5, 4
PREP_ORDER = ['raw', 'snv', 'msc', 'sg_smooth', 'sg_deriv1']
SG_WINDOW, SG_POLYORDER = 11, 2
BATCH_SIZE, LEARNING_RATE, DROPOUT = 32, 1e-3, 0.20
MAX_INNER_EPOCHS, PATIENCE, MIN_DELTA = 250, 20, 0.001
FINAL_SEEDS = [2026, 2027, 2028]
BOOT_REPS, BOOT_SEED = 10000, 20260915
OUTER_FOLDS = list(range(1, N_OUTER + 1))
PREPS = list(PREP_ORDER)
NEW_VARIANTS = ['CNN1D_FLAT', 'CNN1D_GAP_POS']
SEED_BASE = {'CNN1D_FLAT': 62000, 'CNN1D_GAP_POS': 63000}   # inner seed = base + 100*outer + 10*prep_index + inner
REF = 'CNN1D'                                               # NB11 reference (GAP) name used in NB12 files

if QUICK_TEST:
    MAX_INNER_EPOCHS, FINAL_SEEDS, BOOT_REPS = 3, [2026], 300
    PREPS = ['raw', 'sg_deriv1']
    print('*** QUICK_TEST ACTIVE: tiny epochs, 1 seed, 2 preprocessings. Results are NOT valid. ***')

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPUs:', [g.name for g in gpus] or 'none (CPU)')
print('Project root:', PROJECT_ROOT, '| exists:', PROJECT_ROOT.exists())

Mounted at /content/drive
TensorFlow 2.20.0 | Keras 3.13.2 | GPUs: ['/physical_device:GPU:0']
Project root: /content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026 | exists: True


In [2]:
# ---------------------------- integrity gate ----------------------------
def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def find_one(filename, root=PROJECT_ROOT, required=True):
    hits = sorted(root.rglob(filename))
    if not hits:
        assert not required, f'Not found anywhere under {root}: {filename}'
        return None
    if len(hits) > 1:
        print(f'  note: {len(hits)} copies of {filename}; using {hits[0].relative_to(root)}')
    return hits[0]

for p in [DATA_FILE, OUTER_FILE, SPLIT_MANIFEST_FILE]:
    assert p.exists(), f'Missing required input: {p}'
dataset_sha = sha256_file(DATA_FILE)
split_sha = sha256_file(SPLIT_MANIFEST_FILE)
if STRICT_INTEGRITY:
    assert dataset_sha == EXPECTED_DATASET_SHA256, 'Dataset hash changed.'
    assert split_sha == EXPECTED_SPLIT_MANIFEST_SHA256, 'Frozen split manifest hash changed.'
    manifest = json.loads(SPLIT_MANIFEST_FILE.read_text(encoding='utf-8'))
    for fname, expected in manifest['files'].items():
        fp = SPLIT_DIR / fname
        assert fp.exists() and sha256_file(fp) == expected, f'Frozen split changed: {fname}'
    print('PASS - dataset and frozen splits verified by SHA-256.')
else:
    print('WARNING: integrity gate skipped (STRICT_INTEGRITY=False).')

# reference inputs produced by earlier notebooks (searched by name; no hard-coded sub-folders)
NB11_SELECTED_FILE = find_one('NB11_selected_configurations.csv')
NB11_SEEDWISE_FILE = find_one('NB11_oof_predictions_seedwise.csv')
NB12_PEREGG_FILE   = find_one('NB12_per_egg_MAE_wide.csv')
NB12_UNIFIED_FILE  = find_one('NB12_unified_oof_predictions.csv')
NB09A_ROWLEVEL_FILE = find_one('NB09A_rowlevel_oof_seedmean.csv', required=RUN_PART_B)
NB09A_COMPARISON_FILE = find_one('NB09A_ROWLEVEL_VS_EGGDISJOINT_COMPARISON.csv', required=False)
NB05_MAP_FILE  = find_one('NB05_wavelength_order_map.csv', required=RUN_PART_C)
NB05B_MAP_FILE = find_one('NB05B_wavelength_order_map.csv', required=RUN_PART_C)
print('Reference inputs located.')

PASS - dataset and frozen splits verified by SHA-256.
  note: 2 copies of NB11_selected_configurations.csv; using 05_RESULTS/REVISION_REVIEWERS_2026_09/NB11_CNN1D_REVIEWER_BENCHMARK/NB11_selected_configurations.csv
  note: 2 copies of NB11_oof_predictions_seedwise.csv; using 05_RESULTS/REVISION_REVIEWERS_2026_09/NB11_CNN1D_REVIEWER_BENCHMARK/NB11_oof_predictions_seedwise.csv
  note: 2 copies of NB12_per_egg_MAE_wide.csv; using 05_RESULTS/REVISION_REVIEWERS_2026_09/NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE/NB12_per_egg_MAE_wide.csv
  note: 2 copies of NB12_unified_oof_predictions.csv; using 05_RESULTS/REVISION_REVIEWERS_2026_09/NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE/NB12_unified_oof_predictions.csv
  note: 2 copies of NB05_wavelength_order_map.csv; using 05_RESULTS/NB05_WAVELENGTH_ORDER_ABLATION/NB05_wavelength_order_map.csv
  note: 2 copies of NB05B_wavelength_order_map.csv; using 05_RESULTS/NB05B_MULTISEED_SHUFFLE_ABLATION/NB05B_wavelength_order_map.csv
Reference inputs located.


In [3]:
# ------------------ data, frozen splits, preprocessing (identical to NB11/NB15) ------------------
df = pd.read_csv(DATA_FILE)
outer = pd.read_csv(OUTER_FILE)
spec_cols = sorted([c for c in df.columns if c.startswith('Spectra_')], key=lambda c: float(c.replace('Spectra_', '')))
X_all = df[spec_cols].to_numpy(dtype=np.float32)
y_all = df['storage_days'].to_numpy(dtype=np.float32)
samples = df['sample'].to_numpy()
days = df['storage_days'].to_numpy()
assert X_all.shape == (660, 331)
N_FEATURES = X_all.shape[1]
EGGS = sorted(df['sample'].unique())
assert len(EGGS) == 30 and all((df['sample'] == e).sum() == 22 for e in EGGS)

outer_of_egg = dict(zip(outer['sample'], outer['outer_fold']))
inner_maps = {f: pd.read_csv(SPLIT_DIR / f'inner_group_assignment_outer{f:02d}.csv') for f in range(1, N_OUTER + 1)}
for f in range(1, N_OUTER + 1):
    test_eggs = {e for e, o in outer_of_egg.items() if o == f}
    tr_eggs = set(inner_maps[f]['sample'])
    assert len(test_eggs) == 6 and len(tr_eggs) == 24 and not (test_eggs & tr_eggs)
    assert sorted(inner_maps[f]['inner_fold'].unique()) == list(range(1, N_INNER + 1))
print('Frozen splits OK: 5 outer folds (24 train / 6 test eggs), 4 inner folds each, zero overlap.')

class Prep:
    def __init__(self, name):
        self.name, self.reference_, self.scaler_ = name, None, None
    def _base(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.name == 'raw': return X.copy()
        if self.name == 'snv':
            mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, ddof=1, keepdims=True)
            return (X - mu) / np.where(sd < 1e-12, 1.0, sd)
        if self.name == 'msc':
            ref = self.reference_; rm = ref.mean(); rc = ref - rm; den = np.dot(rc, rc)
            out = np.empty_like(X)
            for i, x in enumerate(X):
                xm = x.mean(); b = np.dot(rc, x - xm) / den
                b = 1.0 if abs(b) < 1e-12 else b
                out[i] = (x - (xm - b * rm)) / b
            return out
        if self.name == 'sg_smooth':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=0, axis=1, mode='interp')
        if self.name == 'sg_deriv1':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=1, delta=1.0, axis=1, mode='interp')
        raise ValueError(self.name)
    def fit(self, X):
        if self.name == 'msc': self.reference_ = np.asarray(X, dtype=np.float64).mean(axis=0)
        self.scaler_ = StandardScaler().fit(self._base(X)); return self
    def transform(self, X):
        return self.scaler_.transform(self._base(X)).astype(np.float32)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

def rows_of(eggs):
    return df['sample'].isin(set(eggs)).to_numpy()

Frozen splits OK: 5 outer folds (24 train / 6 test eggs), 4 inner folds each, zero overlap.


In [4]:
# ------------------------------ models -----------------------------------
POS_CHANNEL = np.linspace(-1.0, 1.0, N_FEATURES, dtype=np.float32)   # fixed, target-independent

def build_model(variant):
    n_ch = 2 if variant == 'CNN1D_GAP_POS' else 1
    inp = keras.Input(shape=(N_FEATURES, n_ch))
    if variant in ('CNN1D', 'CNN1D_GAP', 'CNN1D_GAP_POS'):
        x = layers.Conv1D(16, 7, padding='same')(inp); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(32, 5, padding='same')(x); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.GlobalAveragePooling1D()(x)
    elif variant == 'CNN1D_FLAT':
        x = layers.Conv1D(16, 7, padding='same')(inp); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(4)(x)
        x = layers.Conv1D(32, 5, padding='same')(x); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(4)(x)
        x = layers.Flatten()(x)
    else:
        raise ValueError(variant)
    x = layers.Dense(32, activation='relu')(x); x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1)(x)
    m = keras.Model(inp, out, name=variant)
    m.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse',
              metrics=[keras.metrics.MeanAbsoluteError(name='mae')])
    return m

def param_counts(m):
    tr = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    nt = int(sum(np.prod(w.shape) for w in m.non_trainable_weights))
    return tr, nt, tr + nt

def make_input(variant, X2d):
    X = np.asarray(X2d, dtype=np.float32)[..., None]
    if variant == 'CNN1D_GAP_POS':
        pos = np.broadcast_to(POS_CHANNEL[None, :, None], (X.shape[0], N_FEATURES, 1))
        X = np.concatenate([X, pos], axis=-1)
    return np.ascontiguousarray(X)

def set_seeds(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

# parameter audit (also documents that the reference architecture matches NB11 / NB15)
audit = []
for v in ['CNN1D', 'CNN1D_GAP_POS', 'CNN1D_FLAT']:
    keras.backend.clear_session()
    tr, nt, tot = param_counts(build_model(v))
    audit.append({'variant': v, 'trainable': tr, 'non_trainable': nt, 'total': tot})
audit = pd.DataFrame(audit)
audit.to_csv(RESULT_DIR / 'NB16_parameter_count_audit.csv', index=False)
display(audit)
assert (audit.loc[audit.variant == 'CNN1D', ['trainable', 'non_trainable', 'total']].iloc[0].tolist() == [3905, 96, 4001]), \
    'Reference CNN1D does not reproduce the NB11/NB15 parameter count.'

def keras_best_epoch(history_val_mae):
    """Best epoch (1-based) under Keras EarlyStopping logic: improvement only if value < best - min_delta."""
    best, best_ep = np.inf, 0
    for i, v in enumerate(history_val_mae):
        if v < best - MIN_DELTA:
            best, best_ep = v, i
    return best_ep + 1

def train_early_stopping(variant, Xtr, ytr, Xva, yva, seed):
    keras.backend.clear_session(); gc.collect(); set_seeds(seed)
    m = build_model(variant)
    es = keras.callbacks.EarlyStopping(monitor='val_mae', mode='min', patience=PATIENCE, min_delta=MIN_DELTA,
                                       restore_best_weights=True, verbose=0)
    t0 = time.perf_counter()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_INNER_EPOCHS, batch_size=BATCH_SIZE,
              verbose=0, shuffle=True, callbacks=[es])
    val = h.history['val_mae']
    best_epoch = keras_best_epoch(val)
    kb = getattr(es, 'best_epoch', None)
    if kb is not None and int(kb) + 1 != best_epoch:
        print(f'  note: own best_epoch {best_epoch} vs Keras {int(kb)+1} (using Keras value)')
        best_epoch = int(kb) + 1
    pred = m.predict(Xva, verbose=0).ravel()
    return best_epoch, len(val), time.perf_counter() - t0, pred

def fit_fixed_epochs(variant, Xtr, ytr, epochs, seed):
    keras.backend.clear_session(); gc.collect(); set_seeds(seed)
    m = build_model(variant)
    m.fit(Xtr, ytr, epochs=int(epochs), batch_size=BATCH_SIZE, verbose=0, shuffle=True)
    return m

def pooled_metrics(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); e = p - y
    return {'MAE_days': float(np.mean(np.abs(e))), 'RMSE_days': float(np.sqrt(np.mean(e ** 2))),
            'R2': float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)), 'bias_days': float(np.mean(e)),
            'median_AE_days': float(np.median(np.abs(e))),
            'within_1d_pct': float(100 * np.mean(np.abs(e) <= 1)), 'within_2d_pct': float(100 * np.mean(np.abs(e) <= 2)),
            'within_3d_pct': float(100 * np.mean(np.abs(e) <= 3))}
print('Model utilities ready.')

,variant,trainable,non_trainable,total
0,CNN1D,3905,96,4001
1,CNN1D_GAP_POS,4017,96,4113
2,CNN1D_FLAT,23361,96,23457


Model utilities ready.


In [5]:
# ------------------------ statistics helpers (same conventions as NB12) ------------------------
def per_egg_mae_from_long(oof_long, model, value_col='y_pred'):
    d = oof_long[oof_long['model'] == model].copy()
    d['ae'] = (d[value_col] - d['storage_days']).abs()
    return d.groupby('sample')['ae'].mean().sort_index()

def egg_arrays(oof_long, models):
    """Return Y (E,22) and P (M,E,22) sorted by egg and storage day."""
    key = df[['sample', 'storage_days']].copy(); key['ord'] = np.arange(len(key))
    key = key.sort_values(['sample', 'storage_days'])
    eggs = sorted(key['sample'].unique())
    Y = key['storage_days'].to_numpy(float).reshape(len(eggs), 22)
    P = []
    for m in models:
        d = oof_long[oof_long['model'] == m][['sample', 'storage_days', 'y_pred']]
        d = key.merge(d, on=['sample', 'storage_days'], how='left').sort_values(['sample', 'storage_days'])
        assert d['y_pred'].notna().all(), f'missing predictions for {m}'
        P.append(d['y_pred'].to_numpy(float).reshape(len(eggs), 22))
    return eggs, Y, np.stack(P)

def cluster_bootstrap(Y, P, models, reps=BOOT_REPS, seed=BOOT_SEED, chunk=500):
    """Whole-egg bootstrap of pooled MAE/RMSE/R2 and of paired MAE differences."""
    M, E, _ = P.shape
    rng = np.random.default_rng(seed)
    idx_all = rng.integers(0, E, size=(reps, E))
    mae = np.empty((reps, M)); rmse = np.empty((reps, M)); r2 = np.empty((reps, M))
    for s in range(0, reps, chunk):
        idx = idx_all[s:s + chunk]
        Yb = Y[idx]                                   # (b,E,22)
        for k in range(M):
            err = P[k][idx] - Yb
            mae[s:s + chunk, k] = np.abs(err).mean(axis=(1, 2))
            rmse[s:s + chunk, k] = np.sqrt((err ** 2).mean(axis=(1, 2)))
            sst = ((Yb - Yb.mean(axis=(1, 2), keepdims=True)) ** 2).sum(axis=(1, 2))
            r2[s:s + chunk, k] = 1 - (err ** 2).sum(axis=(1, 2)) / sst
    per_egg = np.abs(P - Y[None]).mean(axis=2)        # (M,E)
    rows = []
    for k, m in enumerate(models):
        rows.append({'model': m, **{f'{n}_{q}': float(np.percentile(a[:, k], p)) for n, a in
                     [('MAE', mae), ('RMSE', rmse), ('R2', r2)] for q, p in [('CI_low', 2.5), ('CI_high', 97.5)]}})
    return pd.DataFrame(rows).set_index('model'), per_egg, idx_all

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj

def rank_biserial(d):
    d = np.asarray(d, float); d = d[d != 0]
    if len(d) == 0: return 0.0
    r = stats.rankdata(np.abs(d)); wp = r[d > 0].sum(); wn = r[d < 0].sum()
    return float((wp - wn) / (wp + wn))

def pairwise_table(per_egg_df, models, idx_all=None, model_pos=None):
    """Two-sided paired Wilcoxon on per-egg MAE, Holm within all pairs, rank-biserial r, optional paired bootstrap CI."""
    rows = []
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            a, b = models[i], models[j]
            d = per_egg_df[a].to_numpy() - per_egg_df[b].to_numpy()
            p = stats.wilcoxon(per_egg_df[a], per_egg_df[b], alternative='two-sided').pvalue if np.any(d != 0) else 1.0
            row = {'model_A': a, 'model_B': b, 'mean_MAE_A': per_egg_df[a].mean(), 'mean_MAE_B': per_egg_df[b].mean(),
                   'delta_MAE_A_minus_B': d.mean(), 'p_raw': p, 'rank_biserial_r': rank_biserial(d),
                   'n_eggs_A_lower': int((d < 0).sum())}
            if idx_all is not None:
                boot = d[idx_all].mean(axis=1)
                row['delta_CI_low'], row['delta_CI_high'] = np.percentile(boot, [2.5, 97.5])
            rows.append(row)
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p_raw'].to_numpy()); return t

def friedman_kendall(per_egg_df, models):
    chi2, p = stats.friedmanchisquare(*[per_egg_df[m].to_numpy() for m in models])
    n, k = len(per_egg_df), len(models)
    ranks = per_egg_df[models].rank(axis=1).mean()
    return {'chi2': float(chi2), 'df': k - 1, 'p': float(p), 'kendall_W': float(chi2 / (n * (k - 1))), 'n_eggs': n,
            'k_models': k, 'mean_ranks': ranks.round(2).to_dict()}
print('Statistics helpers ready.')

Statistics helpers ready.


In [6]:
# ------------------------------- learning-curve settings -------------------------------
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression

INCLUDE_LSTM = True          # slowest model; set False to skip
SCALE_EPOCHS = True          # keep gradient updates ~constant when fewer eggs are used
SIZES = [6, 12, 18, 24]      # training eggs per outer fold (24 = all)
N_REPS = 3                   # random subsets per (outer fold, size < 24)
LC_SEED = 2026
MODELS_LC = ['PLSR', 'SVR', 'ANN', 'CNN1D', 'CNN1D_FLAT'] + (['LSTM'] if INCLUDE_LSTM else [])
DL_MODELS = [m for m in MODELS_LC if m not in ('PLSR', 'SVR')]
EPOCH_CAP = None
if QUICK_TEST:
    SIZES, N_REPS, EPOCH_CAP = [6, 24], 1, 2
    print('*** QUICK_TEST: sizes', SIZES, '| 1 repetition | epochs capped at 2. Results are NOT valid. ***')

# frozen selections (final versions) --------------------------------------------------
sel03 = pd.read_csv(find_one('NB03_selected_configurations.csv'))
sel04 = pd.read_csv(find_one('NB04_selected_configurations.csv'))
sel11 = pd.read_csv(NB11_SELECTED_FILE)
sel16 = pd.read_csv(find_one('NB16_selected_configurations.csv', required=('CNN1D_FLAT' in MODELS_LC)))
svr03 = sel03[sel03['model'] == 'SVR'].sort_values('outer_fold')
if not QUICK_TEST:
    assert svr03['C'].eq(1e5).all() and svr03['epsilon'].tolist() == [1.0, 2.0, 2.0, 2.0, 1.0], \
        'NB03_selected_configurations.csv is not the final v3 file (expected C=1e5 and epsilon 1/2/2/2/1). Remove older copies from the project tree.'
    ann04 = sel04[sel04['model'] == 'ANN'].sort_values('outer_fold')
    assert ann04['selected_epoch'].tolist() == [44, 70, 116, 101, 75], 'NB04 selection file differs from the frozen values.'
print('Frozen selections loaded and verified.')

def frozen_config(model, of):
    if model == 'PLSR':
        r = sel03[(sel03['model'] == 'PLSR') & (sel03['outer_fold'] == of)].iloc[0]
        return dict(prep=str(r['preprocessing']), n_components=int(r['n_components']))
    if model == 'SVR':
        r = svr03[svr03['outer_fold'] == of].iloc[0]
        return dict(prep=str(r['preprocessing']), C=float(r['C']), epsilon=float(r['epsilon']), gamma=float(r['gamma']))
    if model in ('ANN', 'LSTM'):
        r = sel04[(sel04['model'] == model) & (sel04['outer_fold'] == of)].iloc[0]
        return dict(prep=str(r['selected_preprocessing']), epochs=int(r['selected_epoch']))
    if model == 'CNN1D':
        r = sel11[sel11['outer_fold'] == of].iloc[0]
        return dict(prep=str(r['preprocessing']), epochs=int(r['selected_epoch']))
    if model == 'CNN1D_FLAT':
        r = sel16[(sel16['variant'] == 'CNN1D_FLAT') & (sel16['outer_fold'] == of)].iloc[0]
        return dict(prep=str(r['preprocessing']), epochs=int(r['selected_epoch']))
    raise ValueError(model)

def build_any(variant):
    if variant == 'ANN':
        inp = keras.Input(shape=(N_FEATURES,))
        x = layers.Dense(64, activation='relu')(inp); x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(32, activation='relu')(x); x = layers.Dropout(DROPOUT)(x)
        m = keras.Model(inp, layers.Dense(1)(x), name='ANN')
    elif variant == 'LSTM':
        inp = keras.Input(shape=(N_FEATURES, 1))
        x = layers.LSTM(64)(inp); x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(32, activation='relu')(x); x = layers.Dropout(DROPOUT)(x)
        m = keras.Model(inp, layers.Dense(1)(x), name='LSTM')
    else:
        return build_model(variant)
    m.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse', metrics=[keras.metrics.MeanAbsoluteError(name='mae')])
    return m

def input_any(variant, X):
    return np.asarray(X, dtype=np.float32) if variant == 'ANN' else make_input(variant, X)

keras.backend.clear_session()
print('ANN / LSTM builders ready. Trainable params ANN =', param_counts(build_any('ANN'))[0], '(expected 23,361)')
assert param_counts(build_any('ANN'))[0] == 23361
if 'LSTM' in MODELS_LC:
    keras.backend.clear_session(); assert param_counts(build_any('LSTM'))[0] == 19009, 'LSTM parameter count differs from NB15 (19,009)'

  note: 3 copies of NB03_selected_configurations.csv; using 05_RESULTS/NB03_CHEMOMETRIC_BASELINES/NB03_selected_configurations.csv
  note: 5 copies of NB04_selected_configurations.csv; using 05_RESULTS/NB04_DEEP_LEARNING_BENCHMARK/NB04_selected_configurations.csv
Frozen selections loaded and verified.
ANN / LSTM builders ready. Trainable params ANN = 23361 (expected 23,361)


In [7]:
# ------------------------------- fits (resumable) -------------------------------
def fit_predict(model, of, eggs_train, eggs_test, seed):
    cfg = frozen_config(model, of)
    tr, te = rows_of(eggs_train), rows_of(eggs_test)
    pp = Prep(cfg['prep']); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
    if model == 'PLSR':
        est = PLSRegression(n_components=min(cfg['n_components'], Xtr.shape[0] - 1)).fit(Xtr, y_all[tr])
        return np.asarray(est.predict(Xte)).ravel()
    if model == 'SVR':
        est = SVR(kernel='rbf', C=cfg['C'], epsilon=cfg['epsilon'], gamma=cfg['gamma']).fit(Xtr, y_all[tr])
        return est.predict(Xte).ravel()
    n_eggs = len(eggs_train)
    epochs = cfg['epochs'] * (24.0 / n_eggs if SCALE_EPOCHS else 1.0)
    epochs = int(max(1, round(epochs)))
    if EPOCH_CAP: epochs = min(epochs, EPOCH_CAP)
    keras.backend.clear_session(); gc.collect(); set_seeds(seed)
    m = build_any(model)
    m.fit(input_any(model, Xtr), y_all[tr], epochs=epochs, batch_size=BATCH_SIZE, verbose=0, shuffle=True)
    return m.predict(input_any(model, Xte), verbose=0).ravel()

jobs = []
for model in MODELS_LC:
    for of in OUTER_FOLDS:
        for n in SIZES:
            for r in (range(1, 2) if n == 24 else range(1, N_REPS + 1)):
                jobs.append((model, of, n, r))
print('Total fits:', len(jobs))
t0 = time.time(); n_done = 0
parts = []
for model, of, n, r in jobs:
    ck = CKPT_DIR / f'lc_{model}_f{of}_n{n}_r{r}.csv'
    if not ck.exists():
        te_eggs = sorted(e for e, o in outer_of_egg.items() if o == of)
        pool = sorted(inner_maps[of]['sample'])
        if n == 24: eggs_train = pool
        else:
            g = np.random.default_rng(70000 + 1000 * of + 10 * n + r)
            eggs_train = sorted(g.choice(pool, size=n, replace=False).tolist())
        assert not (set(eggs_train) & set(te_eggs))
        pred = fit_predict(model, of, eggs_train, te_eggs, LC_SEED)
        te = rows_of(te_eggs)
        pd.DataFrame({'model': model, 'n_train_eggs': n, 'outer_fold': of, 'rep': r, 'sample': samples[te], 'storage_days': days[te],
                      'y_pred': pred}).to_csv(ck, index=False)
    n_done += 1
    if n_done % 10 == 0 or n_done == len(jobs):
        print(f'[{n_done}/{len(jobs)}] last: {model} fold {of} n={n} rep {r} | elapsed {(time.time()-t0)/60:5.1f} min', flush=True)
    parts.append(pd.read_csv(ck))
lc = pd.concat(parts, ignore_index=True)
lc.to_csv(RESULT_DIR / 'NB19_learning_curve_predictions.csv', index=False)
print('Predictions:', lc.shape)

Total fits: 300
[10/300] last: PLSR fold 1 n=24 rep 1 | elapsed   0.0 min
[20/300] last: PLSR fold 2 n=24 rep 1 | elapsed   0.0 min
[30/300] last: PLSR fold 3 n=24 rep 1 | elapsed   0.0 min
[40/300] last: PLSR fold 4 n=24 rep 1 | elapsed   0.0 min
[50/300] last: PLSR fold 5 n=24 rep 1 | elapsed   0.0 min
[60/300] last: SVR fold 1 n=24 rep 1 | elapsed   0.1 min
[70/300] last: SVR fold 2 n=24 rep 1 | elapsed   0.1 min
[80/300] last: SVR fold 3 n=24 rep 1 | elapsed   0.2 min
[90/300] last: SVR fold 4 n=24 rep 1 | elapsed   0.2 min
[100/300] last: SVR fold 5 n=24 rep 1 | elapsed   0.2 min


[110/300] last: ANN fold 1 n=24 rep 1 | elapsed   2.6 min
[120/300] last: ANN fold 2 n=24 rep 1 | elapsed   5.5 min
[130/300] last: ANN fold 3 n=24 rep 1 | elapsed   9.7 min
[140/300] last: ANN fold 4 n=24 rep 1 | elapsed  13.5 min
[150/300] last: ANN fold 5 n=24 rep 1 | elapsed  16.7 min
[160/300] last: CNN1D fold 1 n=24 rep 1 | elapsed  19.9 min
[170/300] last: CNN1D fold 2 n=24 rep 1 | elapsed  22.3 min
[180/300] last: CNN1D fold 3 n=24 rep 1 | elapsed  26.1 min
[190/300] last: CNN1D fold 4 n=24 rep 1 | elapsed  29.8 min
[200/300] last: CNN1D fold 5 n=24 rep 1 | elapsed  33.2 min
[210/300] last: CNN1D_FLAT fold 1 n=24 rep 1 | elapsed  36.1 min
[220/300] last: CNN1D_FLAT fold 2 n=24 rep 1 | elapsed  39.3 min
[230/300] last: CNN1D_FLAT fold 3 n=24 rep 1 | elapsed  42.7 min
[240/300] last: CNN1D_FLAT fold 4 n=24 rep 1 | elapsed  45.8 min
[250/300] last: CNN1D_FLAT fold 5 n=24 rep 1 | elapsed  49.6 min
[260/300] last: LSTM fold 1 n=24 rep 1 | elapsed  55.7 min
[270/300] last: LSTM fold 

In [8]:
# ------------------------------- aggregation, statistics, figure -------------------------------
lc = pd.read_csv(RESULT_DIR / 'NB19_learning_curve_predictions.csv')
lc['ae'] = (lc['y_pred'] - lc['storage_days']).abs()
rng_b = np.random.default_rng(BOOT_SEED)
IDX = rng_b.integers(0, 30, size=(BOOT_REPS, 30))

rows = []; egg_tabs = {}
for model in MODELS_LC:
    for n in SIZES:
        d = lc[(lc['model'] == model) & (lc['n_train_eggs'] == n)]
        pe = d.groupby(['sample', 'rep'])['ae'].mean().groupby('sample').mean().sort_index()      # per-egg MAE averaged over repetitions
        assert len(pe) == 30
        egg_tabs[(model, n)] = pe.to_numpy()
        boot = pe.to_numpy()[IDX].mean(axis=1)
        pooled_by_rep = d.groupby('rep')['ae'].mean()
        yt = d.groupby(['sample', 'storage_days'])['y_pred'].mean().reset_index()
        rows.append({'model': model, 'n_train_eggs': n, 'MAE_days': pe.mean(), 'MAE_CI_low': np.percentile(boot, 2.5), 'MAE_CI_high': np.percentile(boot, 97.5),
                     'SD_across_repetitions': pooled_by_rep.std(ddof=1) if len(pooled_by_rep) > 1 else np.nan, 'n_repetitions': int(d['rep'].nunique()),
                     'R2': 1 - ((yt['y_pred'] - yt['storage_days']) ** 2).sum() / ((yt['storage_days'] - yt['storage_days'].mean()) ** 2).sum()})
lct = pd.DataFrame(rows); lct.to_csv(RESULT_DIR / 'Table_NB19_learning_curve.csv', index=False); display(lct.round(3))

# improvement from the smallest-but-one to the full size (paired over eggs), and comparison with SVR at every size
lo_n = SIZES[len(SIZES) // 2 - 1] if len(SIZES) > 2 else SIZES[0]; hi_n = SIZES[-1]
imp_rows = []
for model in MODELS_LC:
    a, b = egg_tabs[(model, lo_n)], egg_tabs[(model, hi_n)]
    rel = (a.mean() - b.mean()) / a.mean(); bb = ((a[IDX].mean(axis=1) - b[IDX].mean(axis=1)) / a[IDX].mean(axis=1))
    imp_rows.append({'model': model, 'from_n': lo_n, 'to_n': hi_n, 'MAE_from': a.mean(), 'MAE_to': b.mean(), 'relative_reduction': rel,
                     'CI_low': np.percentile(bb, 2.5), 'CI_high': np.percentile(bb, 97.5)})
imp = pd.DataFrame(imp_rows); imp.to_csv(RESULT_DIR / 'Table_NB19_relative_improvement.csv', index=False); display(imp.round(3))

gap_rows = []
for n in SIZES:
    ref = egg_tabs[('SVR', n)]
    for model in DL_MODELS:
        x = egg_tabs[(model, n)]; d = x - ref
        gap_rows.append({'n_train_eggs': n, 'model': model, 'MAE_minus_SVR': d.mean(), 'CI_low': np.percentile(d[IDX].mean(axis=1), 2.5),
                         'CI_high': np.percentile(d[IDX].mean(axis=1), 97.5), 'ratio_to_SVR': x.mean() / ref.mean()})
gap = pd.DataFrame(gap_rows); gap.to_csv(RESULT_DIR / 'Table_NB19_gap_to_SVR_by_size.csv', index=False); display(gap.round(3))

import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.size': 9, 'axes.linewidth': 0.8})
COL = {'PLSR': '#E69F00', 'SVR': '#0072B2', 'ANN': '#009E73', 'CNN1D': '#CC79A7', 'CNN1D_FLAT': '#D55E00', 'LSTM': '#000000'}
MK = {'PLSR': 's', 'SVR': 'o', 'ANN': 'D', 'CNN1D': '^', 'CNN1D_FLAT': 'v', 'LSTM': 'P'}
fig, ax = plt.subplots(figsize=(4.6, 3.3), constrained_layout=True)
for model in MODELS_LC:
    d = lct[lct['model'] == model]
    ax.plot(d['n_train_eggs'], d['MAE_days'], color=COL[model], marker=MK[model], ms=4, lw=1.2, label=model)
    ax.fill_between(d['n_train_eggs'], d['MAE_CI_low'], d['MAE_CI_high'], color=COL[model], alpha=0.12, lw=0)
ax.axhline(5.5, color='#777777', ls=':', lw=0.9); ax.text(SIZES[0], 5.55, 'DummyMean (5.5 d)', fontsize=7, color='#555555')
ax.set_xticks(SIZES); ax.set_xlabel('Training eggs per outer fold'); ax.set_ylabel('MAE on unseen eggs (days)'); ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=7, ncol=2)
fig.savefig(RESULT_DIR / 'NB19_Figure_learning_curve.png', dpi=600, facecolor='white', bbox_inches='tight')
try:
    from PIL import Image
    Image.open(RESULT_DIR / 'NB19_Figure_learning_curve.png').convert('RGB').save(RESULT_DIR / 'NB19_Figure_learning_curve.tif', compression='tiff_lzw', dpi=(600, 600))
except Exception as e: print('TIFF not written:', e)
plt.close(fig)

# wording anchors (numbers only; interpretation is left to the authors)
def f(x, n=3): return f'{x:.{n}f}'
A = []
for model in MODELS_LC:
    r = imp[imp['model'] == model].iloc[0]
    A.append(f'{model}: MAE {f(r.MAE_from)} days with {int(r.from_n)} training eggs and {f(r.MAE_to)} days with {int(r.to_n)} (relative reduction {100*r.relative_reduction:.1f}%, 95% CI {100*r.CI_low:.1f} to {100*r.CI_high:.1f}%).')
last = gap[gap['n_train_eggs'] == SIZES[-1]]
for _, r in last.iterrows():
    A.append(f'With {int(r.n_train_eggs)} training eggs, {r.model} exceeded SVR by {f(r.MAE_minus_SVR)} days (95% CI {f(r.CI_low)} to {f(r.CI_high)}); ratio {f(r.ratio_to_SVR,2)}.')
(RESULT_DIR / 'NB19_wording_anchors.md').write_text('# NB19 wording anchors (generated from saved CSV files)\n\n' + '\n\n'.join(f'{i+1}. {s}' for i, s in enumerate(A)), encoding='utf-8')
print('\n'.join(A))

,model,n_train_eggs,MAE_days,MAE_CI_low,MAE_CI_high,SD_across_repetitions,n_repetitions,R2
0,PLSR,6,3.114,2.756,3.553,0.531,3,0.700
1,PLSR,12,2.458,2.266,2.665,0.012,3,0.782
2,PLSR,18,2.335,2.138,2.580,0.117,3,0.794
3,PLSR,24,2.267,2.084,2.490,NaN,1,0.796
4,SVR,6,3.277,2.951,3.654,0.304,3,0.689
5,SVR,12,2.532,2.295,2.860,0.147,3,0.779
6,SVR,18,2.405,2.212,2.613,0.192,3,0.802
7,SVR,24,2.193,2.024,2.371,NaN,1,0.817
8,ANN,6,4.168,3.405,5.387,0.336,3,0.314
9,ANN,12,2.903,2.556,3.415,0.259,3,0.691


,model,from_n,to_n,MAE_from,MAE_to,relative_reduction,CI_low,CI_high
0,PLSR,12,24,2.458,2.267,0.078,0.044,0.110
1,SVR,12,24,2.532,2.193,0.134,0.062,0.207
2,ANN,12,24,2.903,2.370,0.184,0.123,0.245
3,CNN1D,12,24,4.595,4.486,0.024,-0.022,0.071
4,CNN1D_FLAT,12,24,3.096,2.725,0.120,0.021,0.205
5,LSTM,12,24,4.910,4.765,0.030,0.006,0.053


,n_train_eggs,model,MAE_minus_SVR,CI_low,CI_high,ratio_to_SVR
0,6,ANN,0.891,0.280,1.870,1.272
1,6,CNN1D,1.892,1.620,2.157,1.577
2,6,CNN1D_FLAT,0.834,0.348,1.516,1.255
3,6,LSTM,1.916,1.542,2.264,1.585
4,12,ANN,0.371,0.150,0.620,1.147
5,12,CNN1D,2.063,1.727,2.395,1.815
6,12,CNN1D_FLAT,0.564,0.292,0.890,1.223
7,12,LSTM,2.378,2.014,2.679,1.939
8,18,ANN,0.136,-0.017,0.272,1.057
9,18,CNN1D,2.167,1.885,2.439,1.901


PLSR: MAE 2.458 days with 12 training eggs and 2.267 days with 24 (relative reduction 7.8%, 95% CI 4.4 to 11.0%).
SVR: MAE 2.532 days with 12 training eggs and 2.193 days with 24 (relative reduction 13.4%, 95% CI 6.2 to 20.7%).
ANN: MAE 2.903 days with 12 training eggs and 2.370 days with 24 (relative reduction 18.4%, 95% CI 12.3 to 24.5%).
CNN1D: MAE 4.595 days with 12 training eggs and 4.486 days with 24 (relative reduction 2.4%, 95% CI -2.2 to 7.1%).
CNN1D_FLAT: MAE 3.096 days with 12 training eggs and 2.725 days with 24 (relative reduction 12.0%, 95% CI 2.1 to 20.5%).
LSTM: MAE 4.910 days with 12 training eggs and 4.765 days with 24 (relative reduction 3.0%, 95% CI 0.6 to 5.3%).
With 24 training eggs, ANN exceeded SVR by 0.178 days (95% CI -0.012 to 0.370); ratio 1.08.
With 24 training eggs, CNN1D exceeded SVR by 2.294 days (95% CI 1.881 to 2.682); ratio 2.05.
With 24 training eggs, CNN1D_FLAT exceeded SVR by 0.532 days (95% CI 0.223 to 0.868); ratio 1.24.
With 24 training eggs, LS

In [9]:
protocol = {'notebook': 'NB19_LEARNING_CURVE', 'run_revision': RUN_REVISION, 'quick_test': QUICK_TEST,
            'purpose': 'Empirical check of the small-sample (N=30 eggs) explanation for the recurrent/convolutional results',
            'dataset_sha256': dataset_sha, 'frozen_split_manifest_sha256': split_sha, 'models': MODELS_LC, 'train_sizes_eggs': SIZES,
            'repetitions_per_size': N_REPS, 'subset_seed_rule': '70000 + 1000*outer + 10*n + rep', 'deep_model_seed': LC_SEED,
            'scale_epochs_to_constant_updates': SCALE_EPOCHS, 'configurations': 'inherited per outer fold from NB03/NB04/NB11/NB16; no retuning',
            'test_eggs': 'the 6 frozen outer-test eggs of each fold (identical for every size)', 'bootstrap_replicates': BOOT_REPS,
            'bootstrap_seed': BOOT_SEED, 'predictive_results_modified': False,
            'caveat': 'Frozen hyperparameters were selected with 24 training eggs; smaller subsets are therefore evaluated off-design. No extrapolation beyond 24 eggs is made.'}
(RESULT_DIR / 'NB19_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
(RESULT_DIR / 'NB19_run_summary.json').write_text(json.dumps({'status': 'COMPLETED', 'run_revision': RUN_REVISION,
        'completed_at_utc': datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')
with open(RESULT_DIR / 'environment_packages.txt', 'w', encoding='utf-8') as fh:
    fh.write(f'Python: {sys.version}\nPlatform: {platform.platform()}\nTensorFlow: {tf.__version__}\nKeras: {keras.__version__}\n\n')
    try: fh.write(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, stderr=subprocess.STDOUT))
    except Exception as e: fh.write(f'pip freeze failed: {e}\n')
try: cpuinfo = subprocess.check_output(['lscpu'], text=True, stderr=subprocess.STDOUT)
except Exception as e: cpuinfo = f'lscpu unavailable: {e}'
try: gpuinfo = subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)
except Exception as e: gpuinfo = f'nvidia-smi unavailable: {e}'
(RESULT_DIR / 'hardware_info.txt').write_text(cpuinfo + '\n\n' + gpuinfo, encoding='utf-8')

ZIP_NAME = 'NB19_RESULTS_LEARNING_CURVE' + ('_QUICKTEST' if QUICK_TEST else '')
zip_base = ZIP_DIR / ZIP_NAME
if zip_base.with_suffix('.zip').exists(): zip_base.with_suffix('.zip').unlink()
tmp = RESULT_DIR.parent / '_NB19_zip_staging'
if tmp.exists(): shutil.rmtree(tmp)
shutil.copytree(RESULT_DIR, tmp, ignore=shutil.ignore_patterns('_CHECKPOINT'))
shutil.make_archive(str(zip_base), 'zip', root_dir=tmp); shutil.rmtree(tmp)
print('ZIP created:', zip_base.with_suffix('.zip'), '|', round(zip_base.with_suffix('.zip').stat().st_size / 1024, 1), 'KB')
print('NB19 SUCCESS')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_base.with_suffix('.zip')))

ZIP created: /content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026/05_RESULTS/ZIP_PACKAGES/NB19_RESULTS_LEARNING_CURVE.zip | 1273.5 KB
NB19 SUCCESS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>